# Data Prep — Holding Phone

Bangun dataset training **detection (bbox)** untuk deteksi orang memegang HP, dari `research/datasets/raw/holding_phone/` (Roboflow `detect-hand-holding-phone` v2).

**1 kelas positif:** `0 = holding_phone` (raw bernama `hand_phone`, di-rename ikut konvensi noun positif di CLAUDE.md). Tidak ada kelas negatif — check ini single-kategori (hitung + violation), mirip `ladder_count`/`safety_cone_count`.

Dataset punya **65 gambar background** (label kosong, tanpa HP) — disimpan apa adanya karena membantu YOLO menekan false positive.

**Output:** `research/datasets/processed/holding_phone/{train,valid,test}/{images,labels}` + `data.yaml`.

## Setup

In [ ]:
%matplotlib inline
from __future__ import annotations

import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import yaml
from PIL import Image

NOTEBOOK_DIR = Path.cwd()
RESEARCH_DIR = NOTEBOOK_DIR.parents[1]
AI_DIR       = NOTEBOOK_DIR.parents[2]

RAW1_DIR = RESEARCH_DIR / 'datasets' / 'raw' / 'holding_phone'
PROC_DIR = RESEARCH_DIR / 'datasets' / 'processed' / 'holding_phone'
DATA_YAML = PROC_DIR / 'data.yaml'

CLASS_NAMES = ['holding_phone']           # 1 kelas positif (raw: 'hand_phone' -> rename)
SPLITS = ['train', 'valid', 'test']
SEED = 42

if not RAW1_DIR.exists():
    raise FileNotFoundError(f'Nggak ketemu: {RAW1_DIR}')
print('Raw      :', RAW1_DIR)
print('Processed:', PROC_DIR)

## Konversi raw -> processed

Set `OVERWRITE_PROCESSED = True` untuk rebuild. File di-rename pendek (`s1_t00000.jpg`) supaya total path < 260 char (limit Windows/OneDrive).

In [ ]:
OVERWRITE_PROCESSED = False


def ext(p: Path) -> Path:
    """Windows extended-length path — hindari MAX_PATH 260 char (folder OneDrive + nama file panjang)."""
    return Path('\\\\?\\' + str(p.resolve()))


def poly_to_bbox(coords):
    xs, ys = coords[0::2], coords[1::2]
    xmin, xmax, ymin, ymax = min(xs), max(xs), min(ys), max(ys)
    clamp = lambda v: max(0.0, min(1.0, v))
    return clamp((xmin + xmax) / 2), clamp((ymin + ymax) / 2), clamp(xmax - xmin), clamp(ymax - ymin)


def convert_label(text, class_map=None):
    """Konversi satu file label -> list baris bbox 'cls cx cy w h'. polygon auto -> bbox."""
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        cls = int(parts[0])
        if class_map is not None:
            cls = class_map[cls]
        coords = [float(p) for p in parts[1:]]
        if len(coords) == 4:
            cx, cy, w, h = coords
        elif len(coords) >= 6 and len(coords) % 2 == 0:
            cx, cy, w, h = poly_to_bbox(coords)
        else:
            continue
        if w <= 0 or h <= 0:
            continue
        out.append(f'{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    return out


def build_from(src_dir, class_map, tag):
    """Copy gambar + tulis label dari src_dir ke PROC_DIR.
    File di-rename PENDEK ({tag}_{split[0]}{i:05d}) supaya total path < 260 char
    (limit Windows/OneDrive) — kalau pakai nama Roboflow asli, read/training bakal error."""
    n = {'train': 0, 'valid': 0}
    for split in SPLITS:
        img_src = src_dir / split / 'images'
        lbl_src = src_dir / split / 'labels'
        if not lbl_src.exists():
            continue
        img_dst = PROC_DIR / split / 'images'
        lbl_dst = PROC_DIR / split / 'labels'
        img_dst.mkdir(parents=True, exist_ok=True)
        lbl_dst.mkdir(parents=True, exist_ok=True)
        i = 0
        for lbl in sorted(lbl_src.glob('*.txt')):
            img = None
            for suffix in ('.jpg', '.jpeg', '.png'):
                cand = img_src / (lbl.stem + suffix)
                if cand.exists():
                    img = cand
                    break
            if img is None:
                continue
            lines = convert_label(lbl.read_text(encoding='utf-8'), class_map)
            stem = f'{tag}_{split[0]}{i:05d}'
            ext(lbl_dst / (stem + '.txt')).write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')
            shutil.copy2(ext(img), ext(img_dst / (stem + img.suffix)))
            i += 1
            n[split] += 1
    return n


def write_data_yaml():
    cfg = {'path': str(PROC_DIR.resolve()), 'train': 'train/images', 'val': 'valid/images',
           'test': 'test/images', 'nc': len(CLASS_NAMES), 'names': CLASS_NAMES}
    PROC_DIR.mkdir(parents=True, exist_ok=True)
    DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')


if DATA_YAML.exists() and not OVERWRITE_PROCESSED:
    print('processed/ sudah ada, skip build. Set OVERWRITE_PROCESSED=True untuk rebuild.')
else:
    # raw: kelas 0 (hand_phone) -> 0 (holding_phone). tag 's1'. Label kosong = background (disimpan).
    n = build_from(RAW1_DIR, class_map={0: 0}, tag='s1')
    write_data_yaml()
    print('Build #1 selesai:', n)
    print('data.yaml:', DATA_YAML)

## Profil dataset

Distribusi kelas, jumlah background, dan statistik ukuran bbox.

In [ ]:
def count_dist(split):
    counts = {i: 0 for i in range(len(CLASS_NAMES))}
    d = PROC_DIR / split / 'labels'
    if not d.exists():
        return counts
    for lbl in d.glob('*.txt'):
        for line in lbl.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] = counts.get(int(line.split()[0]), 0) + 1
    return counts

dist = {s: count_dist(s) for s in SPLITS}
dist_df = pd.DataFrame(dist).fillna(0).astype(int)
dist_df.index = [CLASS_NAMES[i] for i in dist_df.index]
dist_df['total'] = dist_df.sum(axis=1)
n_img = {s: len(list((PROC_DIR / s / 'images').glob('*'))) for s in SPLITS}
n_bg = {s: sum(1 for l in (PROC_DIR / s / 'labels').glob('*.txt') if not l.read_text().strip()) for s in SPLITS}
print('Gambar per split   :', n_img)
print('Background (kosong) :', n_bg, '<- gambar tanpa HP, bagus utk tekan false positive')
print()
print(dist_df.to_string())

rows = []
for lbl in (PROC_DIR / 'train' / 'labels').glob('*.txt'):
    for line in lbl.read_text().splitlines():
        p = line.split()
        if len(p) == 5:
            rows.append({'w': float(p[3]), 'h': float(p[4])})
bb = pd.DataFrame(rows)
if len(bb):
    bb['area'] = bb['w'] * bb['h']
    print('\nStatistik bbox (train, relatif):')
    print(bb[['w', 'h', 'area']].describe().round(4).to_string())
    small = (bb['area'] < 0.01).mean() * 100
    print(f'\nBbox kecil (area<1%): {small:.1f}%  -> HP biasanya kecil; kalau >30%, pertimbangkan imgsz=1280')

## Sample gambar + bbox

Sanity check kualitas label setelah konversi.

In [ ]:
N = 6
imgs = sorted((PROC_DIR / 'train' / 'images').glob('*'))
samples = random.Random(SEED).sample(imgs, min(N, len(imgs)))
COLORS = {0: '#d62728', 1: '#1f77b4'}
cols = 3
rows_n = (len(samples) + cols - 1) // cols
fig, axes = plt.subplots(rows_n, cols, figsize=(16, 5 * rows_n))
axes = axes.flatten()
for ax, ip in zip(axes, samples):
    img = Image.open(ip); w, h = img.size; ax.imshow(img)
    lp = PROC_DIR / 'train' / 'labels' / (ip.stem + '.txt')
    if lp.exists():
        for line in lp.read_text().splitlines():
            p = line.split()
            if len(p) != 5:
                continue
            cls = int(p[0]); cx, cy, bw, bh = [float(v) for v in p[1:]]
            x1, y1 = (cx - bw / 2) * w, (cy - bh / 2) * h
            ax.add_patch(patches.Rectangle((x1, y1), bw * w, bh * h, lw=2,
                         edgecolor=COLORS.get(cls, 'lime'), facecolor='none'))
            ax.text(x1, y1 - 4, CLASS_NAMES[cls], fontsize=8, color='white',
                    bbox=dict(boxstyle='round,pad=0.2', fc=COLORS.get(cls, 'lime'), alpha=0.8))
    ax.set_title(ip.name[:40], fontsize=8); ax.axis('off')
for ax in axes[len(samples):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

## Selesai

Dataset siap di `processed/holding_phone/`. Lanjut ke **`02_train.ipynb`**.